In [63]:
import pandas as pd
import numpy as np
import geopandas as gpd
import datetime
import urllib.request 
from shapely.geometry import box

pd.options.display.max_colwidth = 100
pd.options.display.max_rows = 10
pd.options.display.max_columns = 30

# LA County Wildfire Data (January 2025)

## Data Sources

### Los Angeles County
See ArcGIS Viewer of Data Set here: 
https://data.lacounty.gov/datasets/6241d8e277a541a2b3645947d991c35e/explore?location=34.270980%2C-118.419280%2C8.35

Meta/01_/01_/01_/01_/01_/01_/01_/01_/01_/01_/01_data: https://www.arcgis.com/sharing/rest/content/items/6241d8e277a541a2b3645947d991c35e/info/metadata/metadata.xml?format=default&output=html

### Ventura County

We contacted the Ventura County Department of Emergency Services directly, and received a shapefile and details for the sole evacuation alert issued during the study period. 

In [64]:
url = 'https://services.arcgis.com/RmCCgQtiZLDCtblq/arcgis/rest/services/IPAWS_Jan_2025_Fire_Alerts/FeatureServer/2/query?outFields=*&where=1%3D1&f=geojson'
ipaws_raw = gpd.read_file('GEOJSON:' + url)
resp = urllib.request.urlretrieve(url, '../01_data/01_raw/la.geojson')


In [65]:
ventura = gpd.read_file('../01_data/01_raw/oak_park_kenneth')

## Data Processing

### Los Angeles County

There are four times included with the data. We will filter to orders/warnings that were _in effect_ between January 7 and 10. We define _in effect_ here as after the "Effective" time and before the "Expires" time. In cases where the "Effective" time was not specified, it is assumed to be the time that the order was sent. 

In [66]:
ipaws = ipaws_raw.copy()

# Format Time Columns
ipaws['Effective'] = pd.to_datetime(ipaws['Effective'], unit = 'ms', utc = True)
ipaws['Expires'] = pd.to_datetime(ipaws['Expires'], unit = 'ms', utc = True)
ipaws['CreateDate'] = pd.to_datetime(ipaws['CreateDate'], unit = 'ms', utc = True)
ipaws['Sent'] = pd.to_datetime(ipaws['Sent'], unit = 'ms', utc = True)

# Fill in start date where "effective" is missing
ipaws['Effective'] = ipaws['Effective'].fillna(ipaws['Sent'])


In [67]:
# Identify and remove empty columns & single value columns
ipaws = ipaws.drop([col for col in ipaws.columns if ipaws[col].isnull().all()], axis = 1)
ipaws = ipaws.loc[:, ipaws.nunique() > 1]
ipaws = ipaws.loc[:, ~ipaws.columns.str.contains('Spanish')]

In [68]:

# Filter to Dates of Interest
ipaws = ipaws[
    (
        (ipaws["Effective"] >= pd.Timestamp('2025-01-07', tz = 'America/Los_Angeles')) & 
        (ipaws["Effective"] <= pd.Timestamp('2025-01-10', tz = 'America/Los_Angeles'))
    ) | 
    (
        #pd.isna(ipaws['Effective']) | 
        (ipaws["Expires"] >= pd.Timestamp('2025-01-07', tz = 'America/Los_Angeles')) & 
        (ipaws["Expires"] <= pd.Timestamp('2025-01-10', tz = 'America/Los_Angeles'))
    )
]
ipaws = ipaws[ipaws["Sent"] <= pd.Timestamp('2025-01-10', tz = 'America/Los_Angeles')]


We also exclude alerts specifying that a curfew is in effect or that air quality is poor. 

In [69]:

# Filter to Events of Interest
ipaws = ipaws.loc[~ipaws.Instruction.str.contains('CURFEW', na = True, case = False)] # Curfew
ipaws = ipaws.loc[~ipaws.Category.str.match('Health', na = True, case = False)] # Air Quality




We determine whether an alert was an evacuation order or an evacuation warning based on the "Event" type specified in the data set. However, two alerts were not categorized as "Evacuation Immediate" events, but still explicitly instructed residents to "LEAVE NOW" in the body of the message, so we code these as evacuation orders. 

In [70]:
# Determine Type of Alert
ipaws['CleanedType'] = np.select(
    [   
        (ipaws['AlertID'] == 153), # Contained messages saying to evacuate despite non-matching Event code
        (ipaws['AlertID'] == 166), # Contained messages saying to evacuate despite non-matching Event code
        (ipaws.Event.str.match('Evacuation Immediate', na = False)),
        (ipaws.Event.str.match('Fire Warning', na = False)),
        (ipaws.Event.str.match('Local Area Emergency', na = False)),
    ],
    [
        'evacuation',
        'evacuation',
        'evacuation',
        'warning',
        'warning'
    ],
    default = None
)


### Ventura County

We manually add a row here for the Ventura County evacuation warning. Per our email with the county, we know that this warning was in effect from 3:48PM to 7:04PM on January 9, which is entirely within our study period. 

In [71]:
ventura['CleanedType'] = 'warning'
ventura['Effective'] = pd.Timestamp('2025-01-09 15:48', tz = 'America/Los_Angeles')
ventura['Expires'] = pd.Timestamp('2025-01-09 19:04', tz = 'America/Los_Angeles')
ventura['geometry'] = ventura.geometry.to_crs(ipaws.geometry.crs)

ipaws = pd.concat([ipaws, ventura], ignore_index=True)

### Combine

Finally we combine all areas that were issued an evacuation order and all areas that were issued an evacuation warning over the study period. If an area received _both_ warnings and orders, we only included it in the evacuation order area. 

In [72]:
ipaws.explore()

In [73]:
ipaws_dissolved = ipaws.loc[:,['CleanedType', 'geometry']].dissolve(by = 'CleanedType')
ipaws_dissolved = ipaws_dissolved.reset_index()
ipaws_dissolved.loc[ipaws_dissolved['CleanedType'] == 'warning', 'geometry'] = ipaws_dissolved.loc[ipaws_dissolved['CleanedType'] == 'warning', 'geometry']. \
    difference(ipaws_dissolved.loc[ipaws_dissolved['CleanedType'] == 'evacuation', 'geometry'], align = False)
ipaws_dissolved.explore('CleanedType')

In [74]:
pd.options.display.max_colwidth = 250
#pd.options.display.max_rows = None
#pd.options.display.max_columns = None

ipaws

,AlertID,Identifier,Sent,Sender,Effective,Expires,Headline,Description,Instruction,CMAMtext,CMAMlongtext,MsgType,Language,Category,Event,ResponseType,Urgency,Severity,Certainty,EventCode,SenderName,Parameters,AreaDescription,FipsCode,CreateDate,Shape__Area,Shape__Length,geometry,CleanedType
0,100.0,test-lac-949858-1736276341840,2025-01-07 10:59:00+00:00,bcummings@ceooem.lacounty.gov,2025-01-07 18:59:01+00:00,2025-01-08 10:59:00+00:00,Alert from AlertLACounty,LA County Fire Emergency Message: Fast moving wildfire in your area. BE AWARE of your surroundings and MONITOR the situation closely. Follow all instructions from first responders in the field. More information will be posted on alertla.org when ...,LA County Fire Emergency Message: Fast moving wildfire in your area. BE AWARE of your surroundings and MONITOR the situation closely. Follow all instructions from first responders in the field. More information will be posted on alertla.org when ...,LACoFD: Fast moving wildfire. BE AWARE and MONITOR. More on alertla.org.,LA County Fire Emergency Message: Fast moving wildfire in your area. BE AWARE of your surroundings and MONITOR the situation closely. Follow all instructions from first responders in the field. More information will be posted on alertla.org when ...,Alert,es-US,Fire,Local Area Emergency,Monitor,Expected,Severe,Observed,"SAME:LAE, SAME:LAE",Public Alert System,"LACoFD: Fast moving wildfire. BE AWARE and MONITOR. More on alertla.org. , LA County Fire Emergency Message: Fast moving wildfire in your area. BE AWARE of your surroundings and MONITOR the situation closely. Follow all instructions from first re...",Los Angeles,"006037, 006037",2025-01-07 10:59:00+00:00,0.000396,0.090375,"POLYGON ((-118.595 34.10884, -118.59566 34.10874, -118.59764 34.10888, -118.60845 34.10894, -118.60846 34.10712, -118.61068 34.10712, -118.61063 34.1217, -118.61284 34.12171, -118.61283 34.12353, -118.61063 34.12352, -118.61061 34.12902, -118.601...",warning
1,101.0,1898755112770152,2025-01-07 11:13:00+00:00,200277_LA_City_Public_Alerts_City_of_Los_Angeles,2025-01-07 11:13:00+00:00,2025-01-07 15:13:00+00:00,Wildfire Alert: Palisades Fire,None,"Wildfire Alert- Palisades Fire\n\nLAFD: A wildfire is burning at Palisades Drive. Those nearby should get set for a potential evacuation. Monitor local news, LAFD social media, and lafd.org/alerts for updates. Evacuation preparation info here: la...",LAFD:Wildfire- Palisades Drive. Get set for possible evacuation. lafd.org/alerts,"LAFD: A wildfire is burning at Palisades Drive. Those nearby should get set for a potential evacuation. Monitor local news, LAFD social media, and lafd.org/alerts for updates. Evacuation preparation info here: lafd.org/ready-set-go",Alert,es-US,Geo,Fire Warning,None,Immediate,Extreme,Observed,"SAME:FRW, SAME:FRW","200277,LA City Public Alerts,City of Los Angeles","BLOCKCHANNEL:CAPEXCH, BLOCKCHANNEL:NWEM, BLOCKCHANNEL:EAS, EAS-ORG:CIV, LAFD:Wildfire- Palisades Drive. Get set for possible evacuation. lafd.org/alerts, LAFD: A wildfire is burning at Palisades Drive. Those nearby should get set for a potential ...",Los Angeles,"006037, 006037",2025-01-07 11:13:00+00:00,0.004359,0.273662,"POLYGON ((-118.5784 34.038, -118.6023 34.1004, -118.5089 34.0975, -118.5338 34.0336, -118.5784 34.038))",warning
2,102.0,test-lac-950074-1736278218264,2025-01-07 11:30:00+00:00,jfahey@ceooem.lacounty.gov,2025-01-07 19:30:18+00:00,2025-01-08 11:30:00+00:00,Wild Fire - Evacuation Warning,LA County Fire Emergency Message: Wind driven wildfire start in your area. BE AWARE of your surroundings and MONITOR the situation closely. Follow all instructions from first responders in the field. More information will be posted on alertla.org...,LA County Fire Emergency Message: Wind driven wildfire start in your area. BE AWARE of your surroundings and MONITOR the situation closely. Follow all instructions from first responders in the field. More information will be posted on alertla.org...,LACoFD: Evacuati

In [ ]:

ipaws.to_parquet('../01_data/clean/evac_clean.parquet')
ipaws_dissolved.to_parquet('../01_data/clean/evac_clean_aggregated.parquet')

FileNotFoundError: [Errno 2] Failed to open local file '../data/clean/evac_clean.parquet'. Detail: [errno 2] No such file or directory

In [ ]:
ipaws_dissolved.explore('CleanedType')

In [ ]:
ipaws_dissolved_exploded = ipaws_dissolved.explode(index_parts=False, ignore_index=True)
eaton = ipaws_dissolved_exploded.cx[-118.28:-117.9, 34.13:34.27].dissolve(by = 'CleanedType')
eaton.reset_index(inplace=True)
eaton.to_parquet('../01_data/clean/evac_eaton.parquet')

eaton.explore('CleanedType')

In [ ]:
ipaws_dissolved_exploded = ipaws_dissolved.explode(index_parts=False, ignore_index=True)
palisades = ipaws_dissolved_exploded.cx[-118.8:-118.45,34.02: 34.25].dissolve(by = 'CleanedType')
palisades.reset_index(inplace=True)
palisades.to_parquet('../01_data/clean/evac_eaton.parquet')

palisades.explore('CleanedType')